# Case Study — Banking Analytics & Campaign ETL
**Company (context):** NAE Colombia / Claro partnership · **Synthetic data for portfolio demo**

Goal: clean customer extracts, build risk/value segments, and measure multi-channel response.


In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(21)
n = 8000
raw = pd.DataFrame({
    "customer_id": [f"C{i:05d}" for i in range(n)],
    "age": rng.integers(18, 75, n),
    "income": rng.lognormal(15.2, 0.55, n).round(0),
    "product": rng.choice(["Savings", "Credit", "Prepaid"], n, p=[0.45, 0.35, 0.20]),
    "risk_score": np.clip(rng.normal(0.42, 0.18, n), 0, 1).round(3),
    "channel_pref": rng.choice(["SMS", "WhatsApp", "Email"], n, p=[0.34, 0.41, 0.25]),
    "email": rng.choice(["ok", "OK", " missing ", None], n, p=[0.7, 0.15, 0.1, 0.05]),
})
# Ghost / dirty fields similar to real exports
raw["Unnamed: 0"] = range(n)
raw.loc[rng.choice(n, 120, replace=False), "income"] = np.nan
raw.head()


In [ ]:
# ETL-style cleaning
clean = raw.drop(columns=["Unnamed: 0"]).copy()
clean["email"] = clean["email"].astype("string").str.strip().str.lower()
clean["email_ok"] = clean["email"].eq("ok")
clean["income"] = clean["income"].fillna(clean["income"].median())
clean["value_tier"] = pd.qcut(clean["income"], 3, labels=["Low", "Mid", "High"])
clean["risk_band"] = pd.cut(clean["risk_score"], [-0.01, 0.33, 0.66, 1.01], labels=["Low", "Mid", "High"])
clean.head()


In [ ]:
# Campaign response simulation
camp = clean.sample(3000, random_state=21).copy()
base = {"SMS": 0.041, "WhatsApp": 0.057, "Email": 0.033}
camp["response"] = [
    rng.random() < base[ch] * (1.15 if v == "High" else 0.9 if v == "Low" else 1.0)
    for ch, v in zip(camp["channel_pref"], camp["value_tier"])
]
perf = (
    camp.groupby("channel_pref")
        .agg(sent=("customer_id", "count"), responders=("response", "sum"))
)
perf["response_rate"] = (perf["responders"] / perf["sent"]).round(4)
perf


In [ ]:
summary = {
    "customers_cleaned": int(len(clean)),
    "email_valid_share": round(float(clean["email_ok"].mean()), 3),
    "best_channel": perf["response_rate"].idxmax(),
    "best_response_rate": float(perf["response_rate"].max()),
}
summary
